In [1]:
import pandas as pd

baseline = pd.read_csv("../data/baseline.csv")
reports = pd.read_csv("../data/damage_reports.csv")

print("Baseline Data:")
display(baseline)

print("Report Data:")
display(reports)

Baseline Data:


,zone,baseline_condition
0,Zone A,building intact
1,Zone B,road accessible
2,Zone C,bridge functional


Report Data:


,zone,reported_damage,reported_severity
0,Zone A,minor roof damage,minor
1,Zone B,no damage,none
2,Zone C,road debris,moderate


In [2]:
video_findings = pd.DataFrame([
    {"zone": "Zone A", "timestamp": "00:45", "detected_damage": "collapsed building", "severity": "severe"},
    {"zone": "Zone B", "timestamp": "01:20", "detected_damage": "flooded road", "severity": "moderate"},
    {"zone": "Zone C", "timestamp": "02:10", "detected_damage": "bridge damage", "severity": "severe"}
])

display(video_findings)

,zone,timestamp,detected_damage,severity
0,Zone A,00:45,collapsed building,severe
1,Zone B,01:20,flooded road,moderate
2,Zone C,02:10,bridge damage,severe


In [3]:
merged = video_findings.merge(baseline, on="zone").merge(reports, on="zone")

display(merged)

,zone,timestamp,detected_damage,severity,baseline_condition,reported_damage,reported_severity
0,Zone A,00:45,collapsed building,severe,building intact,minor roof damage,minor
1,Zone B,01:20,flooded road,moderate,road accessible,no damage,none
2,Zone C,02:10,bridge damage,severe,bridge functional,road debris,moderate


In [4]:
def validate(row):
    video = row["severity"]
    report = row["reported_severity"]

    if video == report:
        return "VALIDATED"
    elif report == "none" and video != "none":
        return "UNREPORTED DAMAGE"
    elif video in ["severe"] and report in ["minor", "none"]:
        return "UNDERREPORTED"
    else:
        return "MISMATCH"

merged["status"] = merged.apply(validate, axis=1)

display(merged)

,zone,timestamp,detected_damage,severity,baseline_condition,reported_damage,reported_severity,status
0,Zone A,00:45,collapsed building,severe,building intact,minor roof damage,minor,UNDERREPORTED
1,Zone B,01:20,flooded road,moderate,road accessible,no damage,none,UNREPORTED DAMAGE
2,Zone C,02:10,bridge damage,severe,bridge functional,road debris,moderate,MISMATCH


In [5]:
severity_score = {
    "none": 0,
    "minor": 30,
    "moderate": 60,
    "severe": 90
}

def compute_risk(row):
    return severity_score.get(row["severity"], 50)

merged["risk_score"] = merged.apply(compute_risk, axis=1)

display(merged)

,zone,timestamp,detected_damage,severity,baseline_condition,reported_damage,reported_severity,status,risk_score
0,Zone A,00:45,collapsed building,severe,building intact,minor roof damage,minor,UNDERREPORTED,90
1,Zone B,01:20,flooded road,moderate,road accessible,no damage,none,UNREPORTED DAMAGE,60
2,Zone C,02:10,bridge damage,severe,bridge functional,road debris,moderate,MISMATCH,90


In [6]:
merged.to_csv("../outputs/final_results.csv", index=False)
print("Saved to outputs/final_results.csv")

Saved to outputs/final_results.csv


In [7]:
import boto3
import json
import time
import uuid
import pandas as pd
from pathlib import Path
from IPython.display import clear_output

session = boto3.Session()

AWS_REGION = session.region_name
# If needed, force:
# AWS_REGION = "us-east-1"

bedrock_client = session.client("bedrock-runtime")
s3_client = session.client("s3")

S3_BUCKET_NAME = "twelvelabs-bedrock-workshop-workshopbucket-iyzcfy0cvcak"
S3_VIDEOS_PATH = "videos"
S3_EMBEDDINGS_PATH = "embeddings"

aws_account_id = session.client("sts").get_caller_identity()["Account"]

print("AWS Region:", AWS_REGION)
print("AWS Account:", aws_account_id)
print("S3 Bucket:", S3_BUCKET_NAME)

s3_client.head_bucket(Bucket=S3_BUCKET_NAME)
print("✅ S3 bucket connected")

AWS Region: us-east-1
AWS Account: 218719670224
S3 Bucket: twelvelabs-bedrock-workshop-workshopbucket-iyzcfy0cvcak
✅ S3 bucket connected


In [8]:
PROJECT_ROOT = Path("..")

local_video_path = PROJECT_ROOT / "data" / "videos" / "disaster_video.mp4"
video_s3_key = f"{S3_VIDEOS_PATH}/disaster_video.mp4"

s3_client.upload_file(str(local_video_path), S3_BUCKET_NAME, video_s3_key)

video_s3_uri = f"s3://{S3_BUCKET_NAME}/{video_s3_key}"
print("Uploaded:", video_s3_uri)

Uploaded: s3://twelvelabs-bedrock-workshop-workshopbucket-iyzcfy0cvcak/videos/disaster_video.mp4


In [9]:
PEGASUS_MODEL_ID_REGIONS = {
    "us-east-1": "us.twelvelabs.pegasus-1-2-v1:0",
    "us-west-2": "us.twelvelabs.pegasus-1-2-v1:0",
    "eu-west-1": "eu.twelvelabs.pegasus-1-2-v1:0",
    "ap-northeast-2": "apac.twelvelabs.pegasus-1-2-v1:0"
}

PEGASUS_MODEL_ID = PEGASUS_MODEL_ID_REGIONS[AWS_REGION]
print("Pegasus model:", PEGASUS_MODEL_ID)

Pegasus model: us.twelvelabs.pegasus-1-2-v1:0


In [10]:
!pip install opencv-python

In [11]:
import cv2
import math

local_video_path = "../data/videos/disaster_video.mp4"

def get_video_duration_seconds(video_path):
    video = cv2.VideoCapture(video_path)
    fps = video.get(cv2.CAP_PROP_FPS)
    frame_count = video.get(cv2.CAP_PROP_FRAME_COUNT)
    video.release()

    if fps == 0:
        raise ValueError("Could not read video FPS. Check video file.")

    return math.ceil(frame_count / fps)

video_duration_seconds = get_video_duration_seconds(local_video_path)
interval_seconds = 5

intervals = []
for start in range(0, video_duration_seconds, interval_seconds):
    end = min(start + interval_seconds, video_duration_seconds)
    intervals.append(f"{start//60:02d}:{start%60:02d}-{end//60:02d}:{end%60:02d}")

print("Video duration:", video_duration_seconds, "seconds")
print("Intervals:", intervals)

Video duration: 78 seconds
Intervals: ['00:00-00:05', '00:05-00:10', '00:10-00:15', '00:15-00:20', '00:20-00:25', '00:25-00:30', '00:30-00:35', '00:35-00:40', '00:40-00:45', '00:45-00:50', '00:50-00:55', '00:55-01:00', '01:00-01:05', '01:05-01:10', '01:10-01:15', '01:15-01:18']


In [12]:
prompt = f"""
You are analyzing a post-disaster aerial video.

The video duration is {video_duration_seconds} seconds.

Divide the video into these exact 5-second intervals:
{intervals}

For EACH interval, identify visible disaster damage.

Return exactly one JSON object per interval.

Each object must include:
- zone: Zone A, Zone B, or Zone C
- timestamp: interval start and end time
- damaged_entity: building, road, bridge, utility, vegetation, unknown
- detected_damage: short description
- severity: none, minor, moderate, severe, destroyed
- confidence: number between 0 and 1
- evidence_summary: one sentence

Return ONLY a JSON array. No markdown.
"""

request_body = {
    "inputPrompt": prompt,
    "mediaSource": {
        "s3Location": {
            "uri": video_s3_uri,
            "bucketOwner": aws_account_id
        }
    },
    "temperature": 0,
    "responseFormat": {
        "jsonSchema": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "zone": {"type": "string"},
                    "timestamp": {"type": "string"},
                    "damaged_entity": {"type": "string"},
                    "detected_damage": {"type": "string"},
                    "severity": {"type": "string"},
                    "confidence": {"type": "number"},
                    "evidence_summary": {"type": "string"}
                },
                "required": [
                    "zone",
                    "timestamp",
                    "damaged_entity",
                    "detected_damage",
                    "severity",
                    "confidence",
                    "evidence_summary"
                ]
            }
        }
    }
}

response = bedrock_client.invoke_model(
    modelId=PEGASUS_MODEL_ID,
    body=json.dumps(request_body),
    contentType="application/json",
    accept="application/json"
)

response_body = json.loads(response.get("body").read())
pegasus_output = json.loads(response_body["message"])

video_findings = pd.DataFrame(pegasus_output)
display(video_findings)

video_findings.to_csv("../outputs/video_findings.csv", index=False)
print("✅ Saved video findings")

,confidence,damaged_entity,detected_damage,timestamp,evidence_summary,severity,zone
0,0.9,road,Flooded road with murky water,00:00-00:05,The road is completely submerged in floodwater...,severe,Zone A
1,0.9,bridge,Flooded bridge with murky water,00:05-00:10,"The bridge is surrounded by floodwater, indica...",severe,Zone A
2,0.9,bridge,Flooded bridge with murky water,00:10-00:15,"The bridge is surrounded by floodwater, indica...",severe,Zone A
3,0.9,bridge,Flooded bridge with murky water,00:15-00:20,"The bridge is surrounded by floodwater, indica...",severe,Zone A
4,0.9,bridge,Flooded bridge with murky water,00:20-00:25,"The bridge is surrounded by floodwater, indica...",severe,Zone A
5,0.9,road,Flooded road with murky water,00:25-00:30,The road is completely submerged in floodwater...,severe,Zone A
6,0.9,road,Flooded road with murky water,00:30-00:35,The road is completely submerged in floodwater...,severe,Zone A
7,0.9,road,Flooded road with murky water,00:35-00:40,The road is completely submerged in floodwater...,severe,Zone A
8,0.9,road,Flooded road with murky water,00:40-00:45,The road is completely submerged in floodwater...,severe,Zone A
9,0.9,road,Flooded road with murky water,00:45-00:50,The road is completely submerged in floodwater...,severe,Zone A


✅ Saved video findings


In [13]:
MARENGO_MODEL_ID = "twelvelabs.marengo-embed-3-0-v1:0"

MARENGO_INFERENCE_ID_REGIONS = {
    "us-east-1": "us.twelvelabs.marengo-embed-3-0-v1:0",
    "eu-west-1": "eu.twelvelabs.marengo-embed-3-0-v1:0",
    "ap-northeast-2": "apac.twelvelabs.marengo-embed-3-0-v1:0"
}

MARENGO_INFERENCE_ID = MARENGO_INFERENCE_ID_REGIONS[AWS_REGION]
print("Marengo model:", MARENGO_INFERENCE_ID)

Marengo model: us.twelvelabs.marengo-embed-3-0-v1:0


In [14]:
def wait_for_embedding_output(s3_bucket: str, s3_prefix: str, invocation_arn: str):
    status = None

    while status not in ["Completed", "Failed", "Expired"]:
        response = bedrock_client.get_async_invoke(invocationArn=invocation_arn)
        status = response["status"]
        print("Embedding task status:", status)
        time.sleep(5)

    if status != "Completed":
        raise Exception(f"Embedding task failed with status: {status}")

    response = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_prefix)

    for obj in response.get("Contents", []):
        if obj["Key"].endswith("output.json"):
            output_key = obj["Key"]
            obj = s3_client.get_object(Bucket=s3_bucket, Key=output_key)
            content = obj["Body"].read().decode("utf-8")
            return json.loads(content).get("data", [])

    raise Exception("No output.json found")


def create_video_embedding(video_s3_uri: str):
    unique_id = uuid.uuid4()
    s3_output_prefix = f"{S3_EMBEDDINGS_PATH}/{S3_VIDEOS_PATH}/{unique_id}"

    response = bedrock_client.start_async_invoke(
        modelId=MARENGO_MODEL_ID,
        modelInput={
            "inputType": "video",
            "video": {
                "mediaSource": {
                    "s3Location": {
                        "uri": video_s3_uri,
                        "bucketOwner": aws_account_id
                    }
                },
                "embeddingOption": ["visual"],
                "embeddingScope": ["clip"]
            }
        },
        outputDataConfig={
            "s3OutputDataConfig": {
                "s3Uri": f"s3://{S3_BUCKET_NAME}/{s3_output_prefix}"
            }
        }
    )

    invocation_arn = response["invocationArn"]
    print("Video embedding task started:", invocation_arn)

    embedding_data = wait_for_embedding_output(
        S3_BUCKET_NAME,
        s3_output_prefix,
        invocation_arn
    )

    print(f"✅ Marengo created {len(embedding_data)} video embedding segments")
    return embedding_data

In [15]:
video_embedding_data = create_video_embedding(video_s3_uri)

Video embedding task started: arn:aws:bedrock:us-east-1:218719670224:async-invoke/j905mge2qcan
Embedding task status: InProgress
Embedding task status: InProgress
Embedding task status: Completed
✅ Marengo created 10 video embedding segments


In [16]:
baseline = pd.read_csv("../data/baseline.csv")
reports = pd.read_csv("../data/damage_reports.csv")

merged = video_findings.merge(baseline, on="zone", how="left") \
                       .merge(reports, on="zone", how="left")

display(merged)

,confidence,damaged_entity,detected_damage,timestamp,evidence_summary,severity,zone,baseline_condition,reported_damage,reported_severity
0,0.9,road,Flooded road with murky water,00:00-00:05,The road is completely submerged in floodwater...,severe,Zone A,building intact,minor roof damage,minor
1,0.9,bridge,Flooded bridge with murky water,00:05-00:10,"The bridge is surrounded by floodwater, indica...",severe,Zone A,building intact,minor roof damage,minor
2,0.9,bridge,Flooded bridge with murky water,00:10-00:15,"The bridge is surrounded by floodwater, indica...",severe,Zone A,building intact,minor roof damage,minor
3,0.9,bridge,Flooded bridge with murky water,00:15-00:20,"The bridge is surrounded by floodwater, indica...",severe,Zone A,building intact,minor roof damage,minor
4,0.9,bridge,Flooded bridge with murky water,00:20-00:25,"The bridge is surrounded by floodwater, indica...",severe,Zone A,building intact,minor roof damage,minor
5,0.9,road,Flooded road with murky water,00:25-00:30,The road is completely submerged in floodwater...,severe,Zone A,building intact,minor roof damage,minor
6,0.9,road,Flooded road with murky water,00:30-00:35,The road is completely submerged in floodwater...,severe,Zone A,building intact,minor roof damage,minor
7,0.9,road,Flooded road with murky water,00:35-00:40,The road is completely submerged in floodwater...,severe,Zone A,building intact,minor roof damage,minor
8,0.9,road,Flooded road with murky water,00:40-00:45,The road is completely submerged in floodwater...,severe,Zone A,building intact,minor roof damage,minor
9,0.9,road,Flooded road with murky water,00:45-00:50,The road is completely submerged in floodwater...,severe,Zone A,building intact,minor roof damage,minor


In [17]:
from pathlib import Path
import cv2
import pandas as pd

def extract_frames_every_5_sec(video_path, output_dir, prefix):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # delete old frames
    for f in output_dir.glob("*.jpg"):
        f.unlink()

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    duration = int(total_frames / fps)

    saved = []

    for sec in range(0, duration + 1, 5):
        cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
        success, frame = cap.read()

        if success:
            frame_name = f"{prefix}_{sec:04d}.jpg"
            frame_path = output_dir / frame_name
            cv2.imwrite(str(frame_path), frame)

            saved.append({
                "timestamp_seconds": sec,
                "timestamp": f"{sec//60:02d}:{sec%60:02d}",
                "frame_name": frame_name,
                "frame_path": str(frame_path)
            })

    cap.release()

    return pd.DataFrame(saved)

In [21]:
before_df = extract_frames_every_5_sec(
    "../data/videos/before_video.mp4",
    "../data/before_frames",
    "before"
)

display(before_df)
before_df.to_csv("../data/before_frames_index.csv", index=False)

,timestamp_seconds,timestamp,frame_name,frame_path
0,0,00:00,before_0000.jpg,../data/before_frames/before_0000.jpg
1,5,00:05,before_0005.jpg,../data/before_frames/before_0005.jpg
2,10,00:10,before_0010.jpg,../data/before_frames/before_0010.jpg
3,15,00:15,before_0015.jpg,../data/before_frames/before_0015.jpg
4,20,00:20,before_0020.jpg,../data/before_frames/before_0020.jpg
5,25,00:25,before_0025.jpg,../data/before_frames/before_0025.jpg
6,30,00:30,before_0030.jpg,../data/before_frames/before_0030.jpg
7,35,00:35,before_0035.jpg,../data/before_frames/before_0035.jpg
8,40,00:40,before_0040.jpg,../data/before_frames/before_0040.jpg
9,45,00:45,before_0045.jpg,../data/before_frames/before_0045.jpg


In [22]:
after_df = extract_frames_every_5_sec(
    "../data/videos/disaster_video.mp4",
    "../data/after_frames",
    "after"
)

display(after_df)
after_df.to_csv("../data/after_frames_index.csv", index=False)

,timestamp_seconds,timestamp,frame_name,frame_path
0,0,00:00,after_0000.jpg,../data/after_frames/after_0000.jpg
1,5,00:05,after_0005.jpg,../data/after_frames/after_0005.jpg
2,10,00:10,after_0010.jpg,../data/after_frames/after_0010.jpg
3,15,00:15,after_0015.jpg,../data/after_frames/after_0015.jpg
4,20,00:20,after_0020.jpg,../data/after_frames/after_0020.jpg
5,25,00:25,after_0025.jpg,../data/after_frames/after_0025.jpg
6,30,00:30,after_0030.jpg,../data/after_frames/after_0030.jpg
7,35,00:35,after_0035.jpg,../data/after_frames/after_0035.jpg
8,40,00:40,after_0040.jpg,../data/after_frames/after_0040.jpg
9,45,00:45,after_0045.jpg,../data/after_frames/after_0045.jpg


In [23]:
before_df = extract_frames_every_5_sec(
    "../data/videos/before_video.mp4",
    "../data/before_frames",
    "before"
)

after_df = extract_frames_every_5_sec(
    "../data/videos/disaster_video.mp4",
    "../data/after_frames",
    "after"
)

display(before_df)
display(after_df)

,timestamp_seconds,timestamp,frame_name,frame_path
0,0,00:00,before_0000.jpg,../data/before_frames/before_0000.jpg
1,5,00:05,before_0005.jpg,../data/before_frames/before_0005.jpg
2,10,00:10,before_0010.jpg,../data/before_frames/before_0010.jpg
3,15,00:15,before_0015.jpg,../data/before_frames/before_0015.jpg
4,20,00:20,before_0020.jpg,../data/before_frames/before_0020.jpg
5,25,00:25,before_0025.jpg,../data/before_frames/before_0025.jpg
6,30,00:30,before_0030.jpg,../data/before_frames/before_0030.jpg
7,35,00:35,before_0035.jpg,../data/before_frames/before_0035.jpg
8,40,00:40,before_0040.jpg,../data/before_frames/before_0040.jpg
9,45,00:45,before_0045.jpg,../data/before_frames/before_0045.jpg


,timestamp_seconds,timestamp,frame_name,frame_path
0,0,00:00,after_0000.jpg,../data/after_frames/after_0000.jpg
1,5,00:05,after_0005.jpg,../data/after_frames/after_0005.jpg
2,10,00:10,after_0010.jpg,../data/after_frames/after_0010.jpg
3,15,00:15,after_0015.jpg,../data/after_frames/after_0015.jpg
4,20,00:20,after_0020.jpg,../data/after_frames/after_0020.jpg
5,25,00:25,after_0025.jpg,../data/after_frames/after_0025.jpg
6,30,00:30,after_0030.jpg,../data/after_frames/after_0030.jpg
7,35,00:35,after_0035.jpg,../data/after_frames/after_0035.jpg
8,40,00:40,after_0040.jpg,../data/after_frames/after_0040.jpg
9,45,00:45,after_0045.jpg,../data/after_frames/after_0045.jpg


In [24]:
import cv2
import pandas as pd
from pathlib import Path
from skimage.metrics import structural_similarity as ssim

def compare_images(before_path, after_path):
    before = cv2.imread(str(before_path), cv2.IMREAD_GRAYSCALE)
    after = cv2.imread(str(after_path), cv2.IMREAD_GRAYSCALE)

    before = cv2.resize(before, (512, 512))
    after = cv2.resize(after, (512, 512))

    similarity = ssim(before, after)
    change_score = round((1 - similarity) * 100, 2)

    if change_score < 15:
        severity = "No Damage"
    elif change_score < 30:
        severity = "Minor"
    elif change_score < 50:
        severity = "Moderate"
    elif change_score < 70:
        severity = "Severe"
    else:
        severity = "Destroyed"

    return change_score, severity

results = []

for _, before_row in before_df.iterrows():
    sec = before_row["timestamp_seconds"]

    matching_after = after_df[after_df["timestamp_seconds"] == sec]

    if not matching_after.empty:
        after_row = matching_after.iloc[0]

        change_score, severity = compare_images(
            before_row["frame_path"],
            after_row["frame_path"]
        )

        results.append({
            "timestamp_seconds": sec,
            "timestamp": before_row["timestamp"],
            "before_frame": before_row["frame_path"],
            "after_frame": after_row["frame_path"],
            "change_score": change_score,
            "severity": severity
        })

damage_df = pd.DataFrame(results)
display(damage_df)

damage_df.to_csv("../outputs/frame_damage_comparison.csv", index=False)

,timestamp_seconds,timestamp,before_frame,after_frame,change_score,severity
0,0,00:00,../data/before_frames/before_0000.jpg,../data/after_frames/after_0000.jpg,44.05,Moderate
1,5,00:05,../data/before_frames/before_0005.jpg,../data/after_frames/after_0005.jpg,45.18,Moderate
2,10,00:10,../data/before_frames/before_0010.jpg,../data/after_frames/after_0010.jpg,60.77,Severe
3,15,00:15,../data/before_frames/before_0015.jpg,../data/after_frames/after_0015.jpg,53.49,Severe
4,20,00:20,../data/before_frames/before_0020.jpg,../data/after_frames/after_0020.jpg,54.59,Severe
5,25,00:25,../data/before_frames/before_0025.jpg,../data/after_frames/after_0025.jpg,61.53,Severe
6,30,00:30,../data/before_frames/before_0030.jpg,../data/after_frames/after_0030.jpg,56.86,Severe
7,35,00:35,../data/before_frames/before_0035.jpg,../data/after_frames/after_0035.jpg,60.91,Severe
8,40,00:40,../data/before_frames/before_0040.jpg,../data/after_frames/after_0040.jpg,72.51,Destroyed
9,45,00:45,../data/before_frames/before_0045.jpg,../data/after_frames/after_0045.jpg,69.02,Severe


In [25]:
from datetime import datetime
from pathlib import Path

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Add recommended action
def recommended_action(severity):
    if severity in ["Destroyed", "Severe"]:
        return "Immediate field inspection and emergency response prioritization"
    elif severity == "Moderate":
        return "Schedule field validation and monitor closely"
    elif severity == "Minor":
        return "Monitor and document for follow-up"
    else:
        return "No immediate action required"

damage_df["recommended_action"] = damage_df["severity"].apply(recommended_action)

# Save CSV
damage_df.to_csv("../outputs/fema_damage_assessment.csv", index=False)

# Build report
report_lines = []

report_lines.append("FEMA-COMPATIBLE DAMAGE ASSESSMENT REPORT")
report_lines.append("=" * 60)
report_lines.append(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append("Incident Type: Flood / Disaster Damage Assessment")
report_lines.append("Assessment Method: Pre-disaster video frames compared with post-disaster aerial video frames")
report_lines.append("Data Sources: Before video frames, After disaster video frames, image change scoring")
report_lines.append("")

report_lines.append("SUMMARY")
report_lines.append("-" * 60)
report_lines.append(f"Total Assessed Segments: {len(damage_df)}")
report_lines.append(f"No Damage: {len(damage_df[damage_df['severity'] == 'No Damage'])}")
report_lines.append(f"Minor: {len(damage_df[damage_df['severity'] == 'Minor'])}")
report_lines.append(f"Moderate: {len(damage_df[damage_df['severity'] == 'Moderate'])}")
report_lines.append(f"Severe: {len(damage_df[damage_df['severity'] == 'Severe'])}")
report_lines.append(f"Destroyed: {len(damage_df[damage_df['severity'] == 'Destroyed'])}")
report_lines.append("")

report_lines.append("DETAILED FINDINGS")
report_lines.append("-" * 60)

for _, row in damage_df.iterrows():
    report_lines.append(f"Assessment ID: DA-{int(row['timestamp_seconds']):04d}")
    report_lines.append(f"Timestamp: {row['timestamp']}")
    report_lines.append(f"Damage Type: Flood / structural visual change")
    report_lines.append(f"Severity Classification: {row['severity']}")
    report_lines.append(f"Change Score: {row['change_score']}")
    report_lines.append(f"Recommended Action: {row['recommended_action']}")
    report_lines.append(f"Evidence Before Frame: {row['before_frame']}")
    report_lines.append(f"Evidence After Frame: {row['after_frame']}")
    report_lines.append("")

report_text = "\n".join(report_lines)

with open("../outputs/fema_damage_report.txt", "w") as f:
    f.write(report_text)

print(report_text)
print("\nSaved:")
print("../outputs/fema_damage_assessment.csv")
print("../outputs/fema_damage_report.txt")

FEMA-COMPATIBLE DAMAGE ASSESSMENT REPORT
Report Generated: 2026-04-26 07:09:05
Incident Type: Flood / Disaster Damage Assessment
Assessment Method: Pre-disaster video frames compared with post-disaster aerial video frames
Data Sources: Before video frames, After disaster video frames, image change scoring

SUMMARY
------------------------------------------------------------
Total Assessed Segments: 16
No Damage: 0
Minor: 0
Moderate: 5
Severe: 8
Destroyed: 3

DETAILED FINDINGS
------------------------------------------------------------
Assessment ID: DA-0000
Timestamp: 00:00
Damage Type: Flood / structural visual change
Severity Classification: Moderate
Change Score: 44.05
Recommended Action: Schedule field validation and monitor closely
Evidence Before Frame: ../data/before_frames/before_0000.jpg
Evidence After Frame: ../data/after_frames/after_0000.jpg

Assessment ID: DA-0005
Timestamp: 00:05
Damage Type: Flood / structural visual change
Severity Classification: Moderate
Change Score